# Lab 20-07 · Flash, SAR, Pipeline and Interleaving

**Predict first**, then run. The handout is [README.md](README.md). All models are in `nyq.py`; energies are in units of $C_uV_{REF}^2$.

In [1]:
import collections
import math
import re
import shutil
import subprocess
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import nyq


def ngspice(netlist):
    if shutil.which("ngspice") is None:
        print(f"ngspice is not installed: skipping {netlist} (the README lists the expected values)")
        return collections.defaultdict(lambda: float("nan"))
    out = subprocess.run(["ngspice", "-b", netlist], capture_output=True, text=True).stdout
    return {m.group(1): float(m.group(2)) for m in re.finditer(r"^(\w+)\s*=\s*([-+0-9.eE]+)", out, re.M)}

## Part 1 · Bubbles

*Predict* what each encoder outputs for 1111000 and for 1101000 (comparator 1 first).

In [2]:
for s in ("1111000", "1101000"):
    th = np.array([[int(c) for c in s]])
    print(f"{s}: ones count {nyq.ones_count(th)[0]}, first transition {nyq.first_transition(th)[0]}")

1111000: ones count 4, first transition 4
1101000: ones count 3, first transition 2


## Part 2 · SAR switching energy for every code

*Predict* which code costs the most energy with conventional switching, and roughly how much flatter the monotonic curve is.

In [3]:
n = 10
codes = np.arange(2 ** n)
ec = np.array([2 * nyq.conventional_energy(n, (c + 0.5) / 2 ** n) for c in codes])
em = np.array([nyq.monotonic_energy(n, ((c + 0.5) / 2 ** n - 0.5) * 2) for c in codes])
for name, e in (("conventional", ec), ("monotonic", em)):
    print(f"{name:12s}: average {e.mean():.2f}, min {e.min():.2f} (code {e.argmin()}), max {e.max():.2f} (code {e.argmax()})")
print(f"saving {100 * (1 - em.mean() / ec.mean()):.2f} %")
plt.figure(); plt.plot(codes, ec, lw=0.6, label="conventional"); plt.plot(codes, em, lw=0.6, label="monotonic")
plt.xlabel("output code"); plt.ylabel("energy (C_u V_REF^2)"); plt.legend(); plt.show()

conventional: average 1363.33, min 682.67 (code 1022), max 1704.67 (code 0)
monotonic   : average 255.50, min 170.67 (code 0), max 298.17 (code 510)
saving 81.26 %


## Part 3 · Pipeline redundancy and residue gain

*Predict* how many output codes change when every comparator has an offset below $V_{REF}/4$, and what a 1% low interstage gain does.

In [4]:
x = np.linspace(-0.99999, 0.99999, 2000001)
ideal = nyq.pipeline(x)
rng = np.random.default_rng(7)
offs = rng.uniform(-0.2, 0.2, (8, 2))
print("codes changed by offsets within +-0.2 VREF:", int(np.sum(nyq.pipeline(x, offsets=offs) != ideal)))
o3 = offs.copy(); o3[0, 1] = 0.3
print("largest error with one 0.3 VREF offset:", int(np.abs(nyq.pipeline(x, offsets=o3) - ideal).max()), "codes")
for g1 in (0.998, 0.99, 1.01):
    g = np.ones(8); g[0] = g1
    c = (nyq.pipeline(x, gains=g) + 511.5).astype(int)
    cnt = np.bincount(c, minlength=1024).astype(float)
    inner = cnt[2:-2]
    dnl = inner / inner.mean() - 1
    print(f"gain {g1}: DNL min {dnl.min():+.3f}, max {dnl.max():+.3f} LSB; interior missing codes {int(np.sum(inner == 0))}; "
          f"end codes never reached {int(np.sum(cnt[:2] == 0) + np.sum(cnt[-2:] == 0))}")
print(ngspice("mdac.cir"))
print("hand formula:", nyq.mdac_residue(0.3, 0.5, 0.5e-12, 0.5e-12, 0.1e-12, 1000), nyq.mdac_residue(0.3, 0.5, 0.5e-12, 0.5e-12, 0.1e-12, 100))

codes changed by offsets within +-0.2 VREF: 0
largest error with one 0.3 VREF offset: 26 codes


gain 0.998: DNL min -0.255, max +0.001 LSB; interior missing codes 0; end codes never reached 0
gain 0.99: DNL min -1.000, max +0.006 LSB; interior missing codes 4; end codes never reached 4


gain 1.01: DNL min -0.005, max +0.990 LSB; interior missing codes 0; end codes never reached 0
{'r1': 0.09978048, 'r2': 0.09784736}
hand formula: 0.0997804829375374 0.09784735812133072


## Part 4 · Interleaving spurs

*Predict* the spur level and frequency for each mismatch of Example 20-07.7 (two channels, 1 GS/s, 0.8 V input at bin 1001 of 4096).

In [5]:
N, fs, m, amp = 4096, 1e9, 1001, 0.8
cases = {"offset 1 mV": (np.array([0, 1e-3]), np.ones(2), np.zeros(2)),
         "gain 1%": (np.zeros(2), np.array([1, 1.01]), np.zeros(2)),
         "skew 1 ps": (np.zeros(2), np.ones(2), np.array([0, 1e-12]))}
fig, axs = plt.subplots(3, 1, sharex=True, figsize=(6, 6))
for ax, (name, (o, g, s)) in zip(axs, cases.items()):
    x, fin = nyq.ti_capture(2, fs, N, m, amp, o, g, s)
    p = nyq.power_spectrum(x)
    b = N // 2 if "offset" in name else N // 2 - m
    print(f"{name:12s}: spur at {b * fs / N / 1e6:7.2f} MHz, {nyq.spur_dbc(p, m, b):.1f} dBc")
    ax.plot(np.arange(p.size) * fs / N / 1e6, 10 * np.log10(p / p[m] + 1e-30), lw=0.6); ax.set_ylim(-160, 5); ax.set_title(name)
axs[-1].set_xlabel("frequency (MHz)"); plt.show()

offset 1 mV : spur at  500.00 MHz, -61.1 dBc
gain 1%     : spur at  255.62 MHz, -46.1 dBc
skew 1 ps   : spur at  255.62 MHz, -62.3 dBc
